In [1]:
2+2

4

## Set up the Qdrant vector database

In [1]:
import qdrant_client
collection_name = "chat_with_docs"

client = qdrant_client.QdrantClient(
    url="http://localhost:6333",
    port=6333,
)

In [4]:
from llama_index.core import SimpleDirectoryReader

input_dir_path = "./docs"

loader = SimpleDirectoryReader(
    input_dir_path,
    required_exts=[".pdf"],
    recursive=True,
)
docs = loader.load_data()



In [3]:
docs

[Document(id_='9ed1f1a9-98d6-4845-8e24-4dd505146175', embedding=None, metadata={'file_path': '/Users/rajesh/Desktop/rajesh/Archive/teaching/skills_caravan/RAG-sessions/docs/dspy.pdf', 'file_name': 'dspy.pdf', 'file_type': 'application/pdf', 'file_size': 460814, 'creation_date': '2026-07-18', 'last_modified_date': '2026-09-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='%PDF-1.5\n%\n109 0 obj\n<< /Filter /FlateDecode /Length 4648 >>\nstream\nxڭ;ێ6\x15~\tF\ruDݕ}lb1\x19`i#K([7V\x07\x0b~bH\x16N:؝w\ue1fb??}>)v_a{:T$H힎\x7fz\x19\x7f\x7fNA`\x0458ݕ\u05fb\x7f+\x1do|Lh]Tw~\x17A,qK]~\x1f\x06W5TW{UʳC՜\x19\'~vUg1O7G\x04Zq4a4MvF^_^\x05^\x7fLC\x11Φ1

In [5]:

type(docs), len(docs)

(list, 1)

In [6]:
!pip install llama-index-vector-stores-qdrant


In [7]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, ServiceContext, StorageContext


def create_index(documents):
    vector_store = QdrantVectorStore(client=client, collection_name=collection_name)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
    )
    return index


### Load the embedding model and index data

In [8]:
!pip install llama-index-embeddings-huggingface

In [9]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5", trust_remote_code=True)

Settings.embed_model = embed_model
index = create_index(docs)



/Users/rajesh/miniconda3/envs/wed_batch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9884.48it/s]


## Define the prompt template

In [10]:
!pip install llama-index-llms-ollama

  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-index-llms-ollama]


### If you want to run with Local setup the use the below code

In [ ]:
# from llama_index.llms.ollama import Ollama

# llm = Ollama(
#     model="deepseek-r1:latest",
#     request_timeout=600.0,
#     thinking=False
#     )
# Settings.llm = llm


### If you want to use chat GPT API then use the below code


In [32]:
!pip install llama-index-llms-openai

In [35]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)


In [36]:
import os
from dotenv import load_dotenv
load_dotenv()
from llama_index.llms.openai import OpenAI
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    )
Settings.llm = llm

print("using the ChatGPT API")

using the ChatGPT API


### Define the prompt template

In [29]:
from llama_index.core import PromptTemplate

qa_prompt_tmpl_str = (
"Context information is below.\n"
"---------------------\n"
"{context_str}\n"
"---------------------\n"
"Given the context information above I want you to think step by step to answer the query in a crisp manner, incase case you don't know the answer say 'I don't know!'.\n"
"Query: {query_str}\n"
"Answer: "
)

qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

In [23]:
qa_prompt_tmpl

PromptTemplate(metadata={'prompt_type': <PromptType.CUSTOM: 'custom'>}, template_vars=['context_str', 'query_str'], kwargs={}, output_parser=None, template_var_mappings=None, function_mappings=None, template="Context information is below.\n---------------------\n{context_str}\n---------------------\nGiven the context information above I want you to think step by step to answer the query in a crisp manner, incase case you don't know the answer say 'I don't know!'.\nQuery: {query_str}\nAnswer: ")

Hi <name>,

today we have <class> class at <time> PM

In [30]:
from llama_index.core.postprocessor import SentenceTransformerRerank


rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-12-v2", 
    top_n=3
    )

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18885.64it/s]


### Query the document

In [37]:
query_engine = index.as_query_engine(
    similarity_top_k=5,
    node_postprocessors=[rerank],
)
query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)


response = query_engine.query("what exactly is DSPy?")
print(response)

DSPy is a framework designed for building and deploying decision systems using data science principles. It allows users to create, manage, and evaluate decision-making processes by integrating data analysis, machine learning, and business logic. DSPy aims to simplify the development of decision systems, making it easier for data scientists and analysts to implement and iterate on their models.


In [ ]:
str(response)